# MODI-SYNTH · Baseline Reader — Kaggle edition **v2**

**Runs unattended.** Set it going with *Save Version → Save & Run All (Commit)*, close the browser.

### One-time setup
1. *Add Input* → upload `colab_pack.zip` as a Dataset (7.6 MB), or attach it if you already have.
2. *Settings → Accelerator →* **GPU T4 x2**.
3. *Settings → Internet →* **On** (the MoDeTrans images come from Hugging Face).
4. *Save Version → Save & Run All (Commit)*.

### What is new in v2 versus the first Kaggle notebook
| | v1 | v2 |
|---|---|---|
| image width fed to the model | always 6144 px | the widest in that batch, nothing more |
| what CTC was told to read | all 768 slots, even over blank padding | only the slots that actually hold text |
| wasted white space | 43% of the average image | 0% beyond the batch's own ragged edge |
| training budget | 10.5 h | 11 h |
| stop after no improvement | 25 epochs | 30 epochs |
| per-epoch log | 9 numbers | 17 numbers, incl. blank share and predicted-vs-true length |

### Running past 12 hours
Kaggle kills a commit at 12 h. This notebook saves `last.pt` after **every** epoch and reloads it
on start, so the way to reach 400–600 epochs is simply to commit it again:

* commit #1 runs epochs 1 → ~N and stops on the 11 h budget
* *Add Input* → add **this notebook's own previous output** as a dataset, then commit #2
* cell 1 copies `runs/` out of that input into `/kaggle/working`, and cell 13 continues at epoch N+1
  with the optimiser state, the learning-rate schedule and the best score all intact

Nothing is re-learned and the cosine schedule does not restart.

In [ ]:
# Cell 1 - environment, GPU, wall-clock budget, and pick up any earlier session's progress
import os, sys, time, json, re, math, random, csv, glob, shutil, zipfile, pathlib
T_START = time.time()
BUDGET_H = 11.0          # Kaggle kills a commit at 12 h; stop at 11 h so cells 14-15 always run
def elapsed_h(): return (time.time() - T_START) / 3600
def budget_left(): return BUDGET_H - elapsed_h()

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on the GPU: Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
print(f'budget: {BUDGET_H} h of training, then it stops and saves whatever it has')

# ---- RESUME. If a previous commit's output is attached as an input dataset, copy its
# runs/ folder here. fit() then finds last.pt and carries straight on from that epoch.
restored = 0
for cand in glob.glob('/kaggle/input/*/runs/*/last.pt'):
    dst = cand.replace('/kaggle/input/', '/kaggle/working/', 1)
    dst = '/kaggle/working/runs/' + cand.split('/runs/', 1)[1]
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(cand), '*')):
        shutil.copyfile(f, os.path.join(os.path.dirname(dst), os.path.basename(f)))
    ck = torch.load(dst, map_location='cpu', weights_only=False)
    print(f"RESUMING: found {cand}  -> epoch {ck['state']['epoch']}, "
          f"best val CER {ck['state']['best_cer']:.2f}")
    restored += 1
if not restored:
    print('fresh start (no previous run attached as an input)')

In [ ]:
# Cell 2 - every setting in one place. Values follow the published recipes, not guesses:
#   UniLipi 2026 (closest work, covers MODI): one line, height 68, Adam lr 2e-4, weight decay 0.01
#   HTR-VT, Pattern Recognition 2025        : one line, 512x64, max lr 1e-3 + warmup + cosine, batch 64
#   FLAME Sanskrit 2026                     : one line, height 50, Adam, batch 4, clip 5,
#                                             "padded to the maximum width WITHIN EACH BATCH"
#   HF/community CRNN practice              : height 60-96, lr 1e-4..6e-4, batch 2-8, cosine, patience ~20
# lr 3e-4 = HTR-VT's 1e-3 scaled from batch 64 down to batch 4, which also lands on UniLipi's 2e-4.
# Trained in FULL fp32 - mixed precision overflowed and silently killed an earlier run.
OUT = '/kaggle/working'
CFG = dict(
    run_name   = 'B1_seed42', seed = 42,

    strip_h = 64, strip_w = 8192, line_gap = 20,   # strip_w is now only a safety CAP, not the input width
    pad_mode = 'batch',                            # 'batch' = FLAME: pad to the widest sample in the batch
    bucket   = True,                               # group samples of similar width into the same batch
    batch_size = 4, accum = 1,
    lr = 3e-4, weight_decay = 1e-2, warmup_steps = 1000,
    max_epochs = 600, patience = 30, min_delta = 0.05,

    use_lstm = True, use_count = True, use_aug = True, span_mask = True,
    count_weight = 0.4, num_workers = 2,
)
CFG['img_h'], CFG['img_w'] = CFG['strip_h'], CFG['strip_w']
RUN_DIR = f"{OUT}/runs/{CFG['run_name']}"
os.makedirs(RUN_DIR, exist_ok=True)
print(f"strip height {CFG['strip_h']} px, width per batch (cap {CFG['strip_w']}) | run dir: {RUN_DIR}")
print(f"stops at: {CFG['max_epochs']} epochs, or {CFG['patience']} epochs with no gain, or {BUDGET_H} h")

In [ ]:
# Cell 3 - libraries Kaggle does not ship
!pip -q install datasets editdistance sacrebleu 2>&1 | tail -1
print('installed')

In [ ]:
# Cell 4 - find colab_pack.zip anywhere under /kaggle/input and unpack the metadata
DATA = '/kaggle/working/data'
os.makedirs(DATA, exist_ok=True)
need = ['splits.json', 'modetrans_text.csv', 'charset.json']

def find_metadata():
    for root, _, files in os.walk('/kaggle/input'):
        if all(n in files for n in need):          # already-unzipped folder
            return root
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                zipfile.ZipFile(os.path.join(root, f)).extractall(DATA)
                for r2, _, f2 in os.walk(DATA):
                    if all(n in f2 for n in need):
                        return r2
    return None

META = find_metadata()
assert META, ('Could not find the metadata. Add colab_pack.zip as an input dataset '
              '(right panel -> Add Input -> Upload).')
print('metadata:', META, sorted(os.listdir(META))[:6])

In [ ]:
# Cell 5 - MoDeTrans images + our fixed splits and cleaned text
import pandas as pd
from datasets import load_dataset

ds = load_dataset('historyHulk/MoDeTrans', split='train')     # 2,043 rows
names  = list(ds['filename'])
splits = json.load(open(f'{META}/splits.json', encoding='utf-8'))['modetrans']['assignment']
text   = (pd.read_csv(f'{META}/modetrans_text.csv', keep_default_na=False)
            .set_index('image_id')['text_train'].to_dict())

assert len(names) == 2043 and set(names) == set(splits) == set(text), 'dataset / split mismatch'
SPLIT_IDS = {s: [n for n in names if splits[n] == s] for s in ['train', 'val', 'test']}
assert not (set(SPLIT_IDS['train']) & (set(SPLIT_IDS['val']) | set(SPLIT_IDS['test']))), 'leakage!'
print('splits:', {k: len(v) for k, v in SPLIT_IDS.items()})

In [ ]:
# Cell 6 - alphabet: index 0 is the CTC blank, then the 86 symbols, then <FRAC> and <unk>
cs = json.load(open(f'{META}/charset.json', encoding='utf-8'))
VOCAB = ['<blank>'] + cs['symbols'] + ['<FRAC>', '<unk>']
STOI = {c: i for i, c in enumerate(VOCAB)}
NUM_CLASSES = len(VOCAB)
_TOK = re.compile(r'<FRAC>|.', re.S)
encode     = lambda s: [STOI.get(t, STOI['<unk>']) for t in _TOK.findall(s)]
decode_ids = lambda ids: ''.join(VOCAB[i] for i in ids if i > 0)

bad = [n for n in names if decode_ids(encode(text[n])) != text[n]]
assert not bad, f'round-trip failed: {bad[:5]}'
assert sum(t == STOI['<unk>'] for n in SPLIT_IDS['train'] for t in encode(text[n])) == 0
MAX_LEN = max(len(encode(text[n])) for n in names)
print(f'{NUM_CLASSES} classes, longest label {MAX_LEN} characters')

In [ ]:
# Cell 7 - THE KEY STEP. Cut the patch into its text lines, crop each one to its ink,
# and lay them END TO END in a single 64 px tall strip, in reading order.
#
# Why a strip: every system that works on this task (UniLipi, HTR-VT, FLAME, the HF CRNNs) feeds
# the network ONE strip of text ~50-96 px tall and reads it left to right. Our earlier runs stacked
# 7 lines into a 448 px tall image, which nobody does - the model had to work out which band each
# character belonged to, on top of reading it. It never got past CER 76.
#
# TWO BUGS FOUND AND FIXED IN v2, both invisible in v1 because nothing checked for them:
#
# (1) The line finder failed on dense and slanted pages. A page whose ink profile never dips below
#     22% of its peak between lines came back as ONE band covering all 5 lines - and that whole
#     block was then squashed down to 64 px tall, destroying it. 150 patches were affected and
#     one of them had a 246-character label crammed into a 183 px strip. Fixed with (a) skew
#     correction before projecting, and (b) splitting any band taller than ~1.6 lines at its own
#     deepest ink valleys. Patches detected as a single line: 150 -> 16.
#
# (2) v1 said the strip cuts blank space from 87% to 61%. That was measured on the strip BEFORE it
#     was padded out to a fixed 6144 px. The image the model actually received was padded, so it
#     was still ~80% blank. The strip was right; the fixed-width padding undid the gain. Cell 8
#     now pads only to the widest sample in each batch.
import numpy as np, cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SH, SW, GAP = CFG['strip_h'], CFG['strip_w'], CFG['line_gap']

def _prof(g):                      # ink per row
    return (cv2.GaussianBlur(g, (5, 5), 0) < 128).sum(1).astype(np.float32)

def _smooth(p, k=31):
    return cv2.GaussianBlur(p.reshape(-1, 1), (1, k), 0).ravel()

def deskew(g, rng=8.0, step=0.5):
    """Rotate to the angle whose row-ink profile has the sharpest peaks and valleys. That is the
    angle at which the text lines are horizontal - classic projection-profile skew correction.
    The range is +-8 deg: the audit found 84 patches piled up at exactly an earlier +-4 limit,
    which is what a search that is too narrow looks like."""
    h, w = g.shape; best, ba = -1.0, 0.0
    for a in np.arange(-rng, rng + 1e-9, step):
        r = g if a == 0 else cv2.warpAffine(g, cv2.getRotationMatrix2D((w/2, h/2), a, 1.0),
                                            (w, h), flags=cv2.INTER_NEAREST, borderValue=255)
        s = float(np.sum(np.diff(_prof(r)) ** 2))
        if s > best: best, ba = s, a
    if ba == 0: return g, 0.0
    return cv2.warpAffine(g, cv2.getRotationMatrix2D((w/2, h/2), ba, 1.0),
                          (w, h), flags=cv2.INTER_LINEAR, borderValue=255), ba

def _bands(p, thr=0.22, minh=12):
    on = p > p.max() * thr
    out, s = [], None
    for i, v in enumerate(on):
        if v and s is None: s = i
        if not v and s is not None:
            if i - s > minh: out.append((s, i))
            s = None
    if s is not None and len(on) - s > minh: out.append((s, len(on)))
    return out

def _split_at_valleys(p, a, b, k, unit):
    """A band holding k merged lines is cut at its k-1 deepest interior minima, keeping cuts
    at least ~half a line apart so a cut never lands in the middle of a line of text."""
    seg = p[a:b]; cuts = []
    cand = sorted((i for i in range(1, len(seg)-1) if seg[i] <= seg[i-1] and seg[i] <= seg[i+1]),
                  key=lambda i: seg[i])
    for i in cand:
        if len(cuts) >= k - 1: break
        if all(abs(i - c) > 0.55*unit for c in cuts) and min(i, len(seg)-i) > 0.45*unit:
            cuts.append(i)
    e = [0] + sorted(cuts) + [len(seg)]
    return [(a + e[i], a + e[i+1]) for i in range(len(e) - 1)]

# the typical height of one line of Modi in this corpus, measured rather than assumed
_h = []
for _r in list(ds)[:300]:
    _g, _ = deskew(np.array(_r['image'].convert('L')))
    _h += [b - a for a, b in _bands(_smooth(_prof(_g)))]
MED_LINE = float(np.median(_h))
print(f'typical line height, measured over 300 patches: {MED_LINE:.0f} px')

# Tuned by measurement, not taste. Lowering the core threshold from 0.22 to 0.08 alone lifts the
# ink the model receives from 90% to 96%. Growing each line toward its neighbour adds the rest,
# but it also makes the band taller, so the writing shrinks inside the 64 px strip. Measured:
#   cap 0.0 -> 96.1% of ink, text fills 64 of 64 px     cap 0.35 -> 99.5% of ink, text fills 48 px
#   cap 0.2 -> 98.7% of ink, text fills 52 px           cap 0.8  -> 100%  of ink, text fills 45 px
# Past 0.35 the ink gain is under half a percent and the writing keeps getting smaller, so 0.35 is
# the knee of the curve. It also lands the writing at ~48 px, which is where FLAME puts a line (50).
LINE_THR, GROW_CAP = 0.08, 0.35

def find_lines(g):
    """Deskew, find the line cores, split any band holding several lines, then GROW each line
    until it meets its neighbour. The growth step is the third fix the audit forced: with a plain
    0.22 threshold, a median patch threw away 10% of its ink before the model ever saw it - the
    tops and tails of letters, and whole sparse lines such as headers and dates. Those characters
    were still in the label, so the model was being asked to read text that was not in its input,
    which puts a floor under the error rate no amount of training can get past. Measured over 400
    patches: ink reaching the model went from 90% (worst case 67%) to 99.5% (worst case 92%)."""
    g2, ang = deskew(g)
    p = _smooth(_prof(g2)); raw = _prof(g2)
    cores = []
    for a, b in (_bands(p, thr=LINE_THR) or [(0, g2.shape[0])]):
        k = int(round((b - a) / MED_LINE))
        cores += (_split_at_valleys(p, a, b, k, MED_LINE)
                  if (k > 1 and (b - a) > 1.6*MED_LINE) else [(a, b)])
    rows = np.where(raw > 0)[0]
    if not len(rows):
        return g2, cores, ang
    top, bot, lim = int(rows[0]), int(rows[-1]) + 1, GROW_CAP*MED_LINE
    out = []
    for i, (a, b) in enumerate(cores):
        lo = top if i == 0 else (cores[i-1][1] + a)//2        # meet the line above at the valley
        hi = bot if i == len(cores)-1 else (b + cores[i+1][0])//2
        out.append((min(max(lo, int(a - lim)), a), max(min(hi, int(b + lim)), b)))
    return g2, out, ang

def to_strip(pil):
    g = np.array(pil.convert('L'))
    g2, lines, ang = find_lines(g)
    segs = []
    for y0, y1 in lines:
        c = g2[max(0, y0):min(g2.shape[0], y1)]   # the band already covers the full ink
        xs = np.where((c < 128).any(0))[0]
        if len(xs) == 0: continue
        c = c[:, xs[0]:xs[-1] + 1]                       # crop to the ink: no side padding
        segs.append(cv2.resize(c, (max(1, int(c.shape[1] * SH / c.shape[0])), SH)))
    if not segs: segs = [np.full((SH, SH), 255, np.uint8)]
    parts = []
    for i, s in enumerate(segs):
        if i: parts.append(np.full((SH, GAP), 255, np.uint8))   # small gap marks a line break
        parts.append(s)
    strip = np.hstack(parts)
    if strip.shape[1] > SW:                              # too long: scale down to fit the cap
        strip = cv2.resize(strip, (SW, SH))
    canvas = np.full((SH, SW), 255, np.uint8)
    canvas[:, :strip.shape[1]] = strip
    return canvas, strip.shape[1], len(segs), ang        # <-- v2 also returns the TRUE width

CACHE  = f"{OUT}/strip_{SH}x{SW}.npy"
CACHEW = f"{OUT}/strip_{SH}x{SW}_widths.npy"
if os.path.exists(CACHE) and os.path.exists(CACHEW):
    IMGS = np.load(CACHE); WID = np.load(CACHEW); nline = angs = None
else:
    res = [to_strip(r['image']) for r in tqdm(ds, desc='deskew, cut into lines, lay end to end')]
    IMGS  = np.stack([a for a, _, _, _ in res])
    WID   = np.array([w for _, w, _, _ in res], np.int32)
    nline = [n for _, _, n, _ in res]; angs = [a for _, _, _, a in res]
    np.save(CACHE, IMGS); np.save(CACHEW, WID)
IDX = {n: i for i, n in enumerate(names)}
if nline is not None:
    import collections
    print('lines per patch:', dict(sorted(collections.Counter(nline).items())))
    print(f'skew corrected: {sum(abs(a)>0.01 for a in angs)}/{len(angs)} patches, '
          f'median |angle| {np.median([abs(a) for a in angs]):.1f} deg')

# Some patches - the ones written small and densely - end up with a strip that is too narrow for
# the number of characters in their label. Two things go wrong there:
#   * if there are fewer slots than the label needs, CTC cannot align it at all and silently
#     returns ZERO loss, so the patch teaches the model nothing and nothing warns you
#   * even above that floor, CTC wants roughly 3-12 output slots per character to work with
# So any strip below 3.2 slots per character is stretched horizontally until it reaches that.
# Stretching adds no new detail, but it does give CTC the room it needs to place characters.
W_STRIDE_EXPECTED = 4
MIN_SLOTS_PER_CHAR = 3.2
_need = {n: len(encode(text[n])) + sum(a == b for a, b in zip(encode(text[n]), encode(text[n])[1:]))
         for n in names}
fixed = 0
for n in names:
    j = IDX[n]
    slots_wanted = max(int(_need[n] * 1.3), int(len(encode(text[n])) * MIN_SLOTS_PER_CHAR) + 1)
    want = slots_wanted * W_STRIDE_EXPECTED
    if WID[j] >= want or want > SW: continue
    IMGS[j, :, :want] = cv2.resize(IMGS[j][:, :WID[j]], (want, SH))
    WID[j] = want; fixed += 1
if fixed:
    np.save(CACHE, IMGS); np.save(CACHEW, WID)
    print(f'widened {fixed} strips that were too cramped for their own label '
          f'(target: at least {MIN_SLOTS_PER_CHAR} output slots per character)')

ink = np.mean([(IMGS[i, :, :WID[i]] <= 250).mean() for i in range(len(WID))])
print(f'strip width used: median {np.median(WID):,.0f} px, mean {WID.mean():,.0f}, '
      f'90th pct {np.percentile(WID,90):,.0f} | hit the {SW:,} cap: {(WID>=SW).sum()} patches')
print(f'  blank inside the strip itself     : {100*(1-ink):.0f}%  (the spaces between words and lines)')
print(f'  blank if padded to a fixed {SW:,}  : {100*(1-(WID.mean()/SW)*ink):.0f}%  <-- what v1 fed the model')
print(f'  blank with per-batch padding (v2) : about {100*(1-ink):.0f}% plus a small ragged edge')

ex = SPLIT_IDS['train'][0]
plt.figure(figsize=(18, 2)); plt.imshow(IMGS[IDX[ex]][:, :WID[IDX[ex]]], cmap='gray'); plt.axis('off')
plt.title(f'{text[ex][:85]}   [{WID[IDX[ex]]} px wide]'); plt.show()
print('cached', IMGS.shape, f'{IMGS.nbytes/1e9:.2f} GB')

In [ ]:
# Cell 8 - augmentation (training only), dynamic-width batching, and the data loaders.
#
# Two things changed in v2:
#  (a) collate() crops every strip to its own true width, then pads the batch to the WIDEST
#      member of that batch - and returns those widths, so CTC can be told how many output
#      slots each sample really has instead of pretending all of them carry text.
#  (b) a bucketing sampler puts strips of similar width in the same batch, which keeps the
#      ragged edge small. Without it a 900 px strip sharing a batch with a 6000 px one would
#      still be padded to 6000.
from torch.utils.data import Dataset, DataLoader, Sampler

def augment(img, rng):
    H, W = img.shape
    out = img
    if rng.random() < 0.5:                                   # small rotate / shear / scale
        M = cv2.getRotationMatrix2D((W/2, H/2), rng.uniform(-1.5, 1.5), rng.uniform(0.96, 1.04))
        sh = rng.uniform(-0.10, 0.10); M[0,1] += sh; M[0,2] -= sh*H/2
        M[1,2] += rng.uniform(-2, 2)
        out = cv2.warpAffine(out, M, (W, H), flags=cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < 0.5:                                   # stroke thickness
        k = np.ones((2, 2), np.uint8)
        out = cv2.erode(out, k) if rng.random() < 0.5 else cv2.dilate(out, k)
    if rng.random() < 0.3:                                   # elastic distortion
        a, s = rng.uniform(10, 25), rng.uniform(4, 6)
        dx = cv2.GaussianBlur((rng.random((H, W))*2-1).astype(np.float32), (0,0), s)*a
        dy = cv2.GaussianBlur((rng.random((H, W))*2-1).astype(np.float32), (0,0), s)*a
        gx, gy = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
        out = cv2.remap(out, gx+dx, gy+dy, cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < 0.4:                                   # speckle / contrast
        out = np.clip(out.astype(np.float32)*rng.uniform(0.9,1.1)
                      + rng.normal(0, rng.uniform(0,8), out.shape), 0, 255).astype(np.uint8)
    return out

W_MULT = 4          # the encoder divides width by 4; pad to a multiple so the slot count is exact

class DS(Dataset):
    def __init__(s, ids, train, aug): s.ids, s.train, s.aug = ids, train, aug
    def __len__(s): return len(s.ids)
    def __getitem__(s, i):
        n = s.ids[i]; j = IDX[n]
        img = IMGS[j][:, :WID[j]]                            # <-- crop away the dead padding
        if s.train and s.aug:
            img = augment(img, np.random.default_rng(torch.randint(0, 2**31-1, (1,)).item()))
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)
        return x, torch.tensor(encode(text[n]), dtype=torch.long), n

def collate(b):
    xs, ys, ns = zip(*b)
    ws = [x.shape[-1] for x in xs]
    Wmax = int(math.ceil(max(ws) / W_MULT) * W_MULT)
    out = torch.full((len(xs), 1, CFG['strip_h'], Wmax), 1.0)   # 1.0 == white after normalising
    for i, x in enumerate(xs): out[i, :, :, :x.shape[-1]] = x
    return (out, torch.cat(ys), torch.tensor([len(y) for y in ys]),
            torch.tensor(ws, dtype=torch.long), list(ns))

class WidthBucketSampler(Sampler):
    # Sort by strip width, cut into batches, then shuffle the ORDER of the batches.
    # Samples in a batch are near-equal in width, so almost no padding is added; the model
    # still sees batches in a random order every epoch.
    def __init__(s, ids, bs, shuffle, drop_last):
        s.ids, s.bs, s.shuffle, s.drop_last = list(ids), bs, shuffle, drop_last
    def __iter__(s):
        order = sorted(range(len(s.ids)), key=lambda i: (WID[IDX[s.ids[i]]],
                       torch.randint(0, 10**6, (1,)).item() if s.shuffle else 0))
        batches = [order[i:i+s.bs] for i in range(0, len(order), s.bs)]
        if s.drop_last and len(batches) > 1 and len(batches[-1]) < s.bs: batches.pop()
        if s.shuffle:
            perm = torch.randperm(len(batches)).tolist(); batches = [batches[i] for i in perm]
        return iter(batches)
    def __len__(s):
        n = (len(s.ids) + s.bs - 1) // s.bs
        return n - 1 if (s.drop_last and len(s.ids) % s.bs) else n

def make_loader(ids, train, cfg=None):
    cfg = cfg or CFG; bs, nw = cfg['batch_size'], cfg['num_workers']
    dl_kw = dict(num_workers=nw, collate_fn=collate, pin_memory=True, persistent_workers=nw > 0)
    dsx = DS(ids, train, cfg['use_aug'])
    if cfg.get('bucket', True):
        drop = bool(train) and len(ids) > bs
        return DataLoader(dsx, batch_sampler=WidthBucketSampler(ids, bs, train, drop), **dl_kw)
    return DataLoader(dsx, batch_size=bs, shuffle=train,
                      drop_last=train and len(ids) > bs, **dl_kw)

# how much padding the bucketing saves, measured on the real training split
_ws = np.array([WID[IDX[n]] for n in SPLIT_IDS['train']], float)
_sam = WidthBucketSampler(SPLIT_IDS['train'], CFG['batch_size'], True, True)
_bat = [[_ws[i] for i in b] for b in _sam]
_waste = np.mean([1 - np.mean(b) / max(b) for b in _bat])
_fixed = 1 - _ws.mean() / CFG['strip_w']
_area = np.mean([max(b) for b in _bat])
print(f'padding added by width-bucketed batching: {100*_waste:.1f}% of each batch')
print(f'padding added by v1 fixed-{CFG["strip_w"]} padding : {100*_fixed:.0f}%')
print(f'mean padded batch width: {_area:,.0f} px vs v1 always {CFG["strip_w"]:,} px '
      f'-> {CFG["strip_w"]/_area:.1f}x less pixel area per batch, so epochs run faster too')

rng = np.random.default_rng(0)
_j = IDX[SPLIT_IDS['train'][1]]
fig, ax = plt.subplots(2, 1, figsize=(18, 3))
ax[0].imshow(IMGS[_j][:, :min(3000, WID[_j])], cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(augment(IMGS[_j][:, :WID[_j]], rng)[:, :3000], cmap='gray'); ax[1].set_title('augmented'); ax[1].axis('off')
plt.show(); print('data pipeline ready')

In [ ]:
# Cell 9 - VISUAL AUDIT of every edit made to the images. Writes pictures, not just numbers.
#
# This cell exists because both of the bugs above were invisible in the metrics and obvious the
# moment somebody looked at a picture. It renders each stage of the editing for a random sample,
# flags the patches where a stage went wrong, and saves the lot to
# /kaggle/working/image-processing-audit/ so it can be downloaded and shown.
#
# The same audit runs locally and in more depth via scripts/19_image_pipeline_audit.py.
AUD = f'{OUT}/image-processing-audit'
os.makedirs(AUD, exist_ok=True)

def _save(fig, name):
    fig.savefig(f'{AUD}/{name}.png', dpi=100, bbox_inches='tight'); plt.close(fig)

# --- the checks, run over all 2,043 patches. Each is a way a stage can fail.
_slots = WID // W_STRIDE_EXPECTED
_len   = np.array([len(encode(text[n])) for n in names])
_nd    = np.array([_need[n] for n in names])
_spc   = _slots / np.maximum(1, _len)
_nlines = np.array(nline) if nline is not None else np.zeros(len(names), int)
CHECKS = {
    'strip_hit_the_width_cap'  : (WID >= SW,      'squeezed horizontally to fit the cap'),
    'too_few_slots_for_label'  : (_slots < _nd,   'CTC cannot align; it returns ZERO loss, silently'),
    'slots_per_char_below_3'   : (_spc < 3.0,     'below the healthy 3-12 slots per character'),
    'slots_per_char_above_20'  : (_spc > 20.0,    'very sparse; harmless for CTC, just more blanks'),
    'detected_as_single_line'  : (_nlines == 1,   'the line finder may have merged the whole page'),
}
print(f'{"check":28s} {"flagged":>8s}   meaning')
_rows = []
for k, (m, why) in CHECKS.items():
    if m.dtype != bool or m.shape[0] != len(names): continue
    print(f'{k:28s} {int(m.sum()):8d}   {why}')
    _rows.append((k, int(m.sum()), round(100*float(m.mean()), 2), why))
pd.DataFrame(_rows, columns=['check','n_flagged','pct','meaning']).to_csv(f'{AUD}/checks.csv', index=False)
pd.DataFrame({'image_id': names, 'strip_w': WID, 'slots': _slots, 'label_len': _len,
              'slots_per_char': _spc.round(2), 'lines': _nlines}).to_csv(f'{AUD}/stats.csv', index=False)

# --- gallery: a fixed random sample through every stage
_rs = np.random.default_rng(42)
for n in [names[i] for i in _rs.choice(len(names), 6, replace=False)]:
    j = IDX[n]; stem = n.replace('.jpg', '')
    s = IMGS[j][:, :WID[j]]
    f, a = plt.subplots(4, 1, figsize=(16, 5))
    a[0].imshow(s[:, :4200], cmap='gray', aspect='auto'); a[0].axis('off')
    a[0].set_title(f'{n} - stage 06: lines cropped, scaled to 64 px, laid end to end '
                   f'({WID[j]} px)', fontsize=8, loc='left')
    _p = np.full((CFG['strip_h'], SW), 255, np.uint8); _p[:, :WID[j]] = s
    a[1].imshow(_p, cmap='gray', aspect='auto'); a[1].axis('off')
    a[1].set_title(f'stage 07 the OLD way: padded to a fixed {SW:,} px -> '
                   f'{100*(1-WID[j]/SW):.0f}% dead paper that CTC was told to read', fontsize=8, loc='left')
    a[2].imshow(augment(s, np.random.default_rng(7))[:, :4200], cmap='gray', aspect='auto'); a[2].axis('off')
    a[2].set_title('stage 08: one training augmentation (affine, stroke, elastic, speckle)',
                   fontsize=8, loc='left')
    a[3].imshow(np.repeat((s[:, :4200] < 128).any(0)[None, :], 20, 0), cmap='RdYlGn', aspect='auto')
    a[3].axis('off')
    a[3].set_title(f'stage 09: green = a column with ink, red = blank. {_slots[j]} output slots for a '
                   f'{_len[j]}-character label = {_spc[j]:.1f} per character (healthy 3-12)',
                   fontsize=8, loc='left')
    f.tight_layout(); _save(f, f'sample__{stem}__stages_06_to_09')

# --- the worst case of each check, so failures are shown rather than described
for k, (m, why) in CHECKS.items():
    idxs = np.where(m)[0]
    if not len(idxs): continue
    j = int(idxs[np.argmin(_spc[idxs])])
    n = names[j]; s = IMGS[j][:, :WID[j]]
    f, a = plt.subplots(figsize=(16, 1.6))
    a.imshow(s[:, :4200], cmap='gray', aspect='auto'); a.axis('off')
    a.set_title(f'WORST CASE of "{k}" - {n}: {WID[j]} px strip, {_slots[j]} slots, '
                f'{_len[j]}-char label = {_spc[j]:.1f} per char. {why}', fontsize=8, loc='left')
    f.tight_layout(); _save(f, f'outlier__{k}__{n.replace(".jpg","")}')

print(f'\nwrote {len([x for x in os.listdir(AUD) if x.endswith(".png")])} images + checks.csv '
      f'+ stats.csv to {AUD}')

In [ ]:
# Cell 10 - the model. One stride changed from v1 (see below); everything else is identical, so
# the comparison stays fair. We also measure how width maps to output slots (W_STRIDE), so CTC
# can be given a per-sample length instead of being told every image is the same length.
import torch.nn as nn, torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(s, cin, cout, stride, drop=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(True),
                              nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU(True),
                              nn.Conv2d(cout, cout, 3, stride=stride, padding=1), nn.ReLU(True),
                              nn.InstanceNorm2d(cout, affine=True), nn.Dropout2d(drop))
    def forward(s, x): return s.net(x)

class DSCBlock(nn.Module):
    def __init__(s, c, drop=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1), nn.ReLU(True),
                              nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1), nn.ReLU(True),
                              nn.InstanceNorm2d(c, affine=True), nn.Dropout2d(drop))
    def forward(s, x): return x + s.net(x)

# The ONE change to the network in v2, and the audit is what justified it. At width/8, 29% of
# patches had fewer than 3 output slots per character and 142 had fewer than 2 - below what CTC
# can align at all. Halving the width stride moves the median from 4.2 to 8.4 slots per
# character and leaves 16 patches short instead of 589. Height is untouched, so the strip
# still collapses to a single row of frames.
class Encoder(nn.Module):            # height /64 : 64 px strip -> ONE output row; width /4
    def __init__(s, d=256):
        super().__init__()
        s.net = nn.Sequential(ConvBlock(1,32,(1,1)), ConvBlock(32,64,(2,2)), ConvBlock(64,128,(2,2)),
                              ConvBlock(128,d,(2,1)), ConvBlock(d,d,(4,1)), ConvBlock(d,d,(2,1)),   # (2,2)->(2,1): width /4, not /8
                              DSCBlock(d), DSCBlock(d), DSCBlock(d), DSCBlock(d))
    def forward(s, x): return s.net(x)

def span_mask(f, ratio=0.1, max_span=8):
    B, D, H, W = f.shape
    m = torch.ones(B, 1, 1, W, device=f.device, dtype=f.dtype)
    for b in range(B):
        for _ in range(max(1, int(W*ratio/((max_span+1)/2)))):
            L = random.randint(1, max_span); st = random.randint(0, W-L); m[b,:,:,st:st+L] = 0
    return f * m

class Reader(nn.Module):
    def __init__(s, n_classes, use_lstm=True, use_count=True, use_span_mask=True, d=256):
        super().__init__()
        s.enc = Encoder(d); s.use_span_mask = use_span_mask
        s.lstm = nn.LSTM(d, d//2, 2, bidirectional=True, batch_first=True, dropout=0.1) if use_lstm else None
        s.head = nn.Conv2d(d, n_classes, 1)
        s.count = nn.Sequential(nn.Linear(d,128), nn.ReLU(True), nn.Linear(128,1)) if use_count else None
    def forward(s, x):
        f = s.enc(x)
        if s.training and s.use_span_mask: f = span_mask(f)
        if s.lstm is not None:
            B, D, H, W = f.shape
            r, _ = s.lstm(f.permute(0,2,3,1).reshape(B*H, W, D))
            f = r.reshape(B,H,W,D).permute(0,3,1,2)
        lo = s.head(f); B, Cc, H, W = lo.shape
        seq = lo.permute(0,2,3,1).reshape(B, H*W, Cc)          # bands top-to-bottom = reading order
        cnt = s.count(f.mean((2,3))).squeeze(1) if s.count is not None else None
        return seq, cnt

DEV = 'cuda'
_m = Reader(NUM_CLASSES, CFG['use_lstm'], CFG['use_count'], CFG['span_mask']).to(DEV).eval()
with torch.no_grad():
    _f = _m.enc(torch.zeros(1, 1, CFG['strip_h'], 512, device=DEV))
ROWS, W_STRIDE = _f.shape[2], 512 // _f.shape[3]
print(f'params {sum(p.numel() for p in _m.parameters())/1e6:.1f} M')
assert ROWS == 1, f'{ROWS} output rows - the strip must collapse to a single row of frames'
assert W_STRIDE == 4, f'width stride is {W_STRIDE}, but collate() pads to multiples of 4'
print(f'grid: 1 row, one output slot per {W_STRIDE} px of width')

# Per sample the number of slots is width/8. CTC needs at least one slot per label character,
# plus one extra wherever the label repeats a character. Check EVERY patch, not just the mean.
_len  = np.array([len(encode(text[n])) for n in names])
_need = np.array([len(encode(text[n])) + sum(a == b for a, b in zip(encode(text[n]), encode(text[n])[1:]))
                  for n in names])
_slots = WID // W_STRIDE
_tight = _slots < _need
print(f'label length: median {np.median(_len):.0f} chars, max {_len.max()}')
print(f'slots per patch: median {np.median(_slots):.0f}, min {_slots.min()} | '
      f'slots per character: median {np.median(_slots/np.maximum(1,_len)):.1f}x  (healthy is 3-12x)')
assert not _tight.any(), (f'{_tight.sum()} patches have fewer slots than their label needs - '
                          f'CTC would silently return 0 loss for them')
print('every patch has room for its label  |  model OK')
del _m, _f; torch.cuda.empty_cache()

In [ ]:
# Cell 11 - decoding and metrics. Metrics are untouched so numbers stay comparable with v1.
# greedy_decode now takes each sample's slot count and ignores slots past it, so a short
# strip in a wide batch cannot pick up stray characters from its own padding.
import editdistance, sacrebleu
_FR = ''
_u = lambda s: s.replace('<FRAC>', _FR)

def greedy_decode(seq, lens=None):
    ids = seq.argmax(-1).cpu().numpy()
    keep = np.ones_like(ids, dtype=bool); keep[:,1:] = ids[:,1:] != ids[:,:-1]; keep &= ids != 0
    if lens is not None:
        for i, L in enumerate(lens): keep[i, int(L):] = False        # drop the padded tail
    return [decode_ids(r[k]) for r, k in zip(ids, keep)]

def blank_share(seq, lens):
    ids = seq.argmax(-1).cpu().numpy()
    tot = hit = 0
    for i, L in enumerate(lens):
        L = int(L); tot += L; hit += int((ids[i, :L] == 0).sum())
    return hit / max(1, tot)

def compute_metrics(refs, hyps, n_boot=0, seed=0):
    e = np.array([editdistance.eval(_u(r), _u(h)) for r, h in zip(refs, hyps)], float)
    L = np.array([max(1, len(_u(r))) for r in refs], float)
    we = np.array([editdistance.eval(r.split(), h.split()) for r, h in zip(refs, hyps)], float)
    wl = np.array([max(1, len(r.split())) for r in refs], float)
    per = e / L
    out = dict(CER=100*e.sum()/L.sum(), WER=100*we.sum()/wl.sum(),
               median_CER=100*float(np.median(per)), catastrophic_pct=100*float((per>0.5).mean()),
               chrF_pp=sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
               BLEU=sacrebleu.corpus_bleu(hyps, [refs]).score, n=len(refs))
    if n_boot:
        rg = np.random.default_rng(seed); idx = rg.integers(0, len(refs), (n_boot, len(refs)))
        b = 100*e[idx].sum(1)/L[idx].sum(1)
        out['CER_CI95'] = (float(np.percentile(b,2.5)), float(np.percentile(b,97.5)))
    return out
print('metrics self-test:', {k: round(v,2) for k,v in compute_metrics(['राम आला'],['राम आल']).items() if k!='n'})

In [ ]:
# Cell 12 - training loop. Every guard here exists because something went wrong without it:
#   * FULL fp32. The first Kaggle run trained well for 12 epochs (CER 100 -> 79.6) and then died:
#     weights grew until the fp16 forward pass overflowed, so from epoch 15 every batch produced a
#     non-finite loss and NOTHING trained for the next 45 minutes.
#   * if more than 5% of an epoch's batches are skipped it ABORTS at once instead of burning hours
#   * logs gradient and weight norms so instability is visible before it is fatal
#   * steps the schedule only when the optimiser actually stepped (removes the fp16 warning)
#   * "cold" is judged by LOSS, not CER: a model printing spaces scores CER ~80 and looks busy
#   * the cold-start abort also requires best CER > 90, because it once killed a run at epoch 34
#     that was at CER 76 and still falling
#   * obeys the wall-clock budget, and saves last.pt every epoch so the next commit resumes
#
# NEW in v2: CTC input_lengths are per-sample (width/8), not "all 768 slots". The model is no
# longer trained to emit blanks across padding it was never going to see at test time.
CTC = nn.CTCLoss(blank=0, zero_infinity=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def slot_lens(ws, n_slots):
    return torch.clamp(torch.as_tensor(ws) // W_STRIDE, min=1, max=n_slots).long()

def loss_fn(seq, cnt, y, yl, ws, cfg):
    logp = seq.float().log_softmax(-1).permute(1, 0, 2)          # (slots, batch, classes)
    il = slot_lens(ws, logp.size(0))                             # <-- the v2 change
    loss = CTC(logp, y, il, yl)
    if cnt is not None:
        loss = loss + cfg['count_weight']*F.l1_loss(cnt.float(), yl.float().to(cnt.device)/100)
    return loss

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); refs, hyps, ns, bl, nb = [], [], [], 0.0, 0
    for x, y, yl, ws, n in loader:
        seq, _ = model(x.to(DEV, non_blocking=True))
        il = slot_lens(ws, seq.shape[1])
        hyps += greedy_decode(seq, il); refs += [text[k] for k in n]; ns += n
        bl += blank_share(seq, il); nb += 1
    return refs, hyps, ns, bl/max(1, nb)

def fit(cfg, train_ids, val_ids, run_dir, verbose_every=1, stop_at_cer=None):
    os.makedirs(run_dir, exist_ok=True); set_seed(cfg['seed'])
    model = Reader(NUM_CLASSES, cfg['use_lstm'], cfg['use_count'], cfg['span_mask']).to(DEV)
    tl = make_loader(train_ids, True, cfg); vl = make_loader(val_ids, False, cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    upd_per_ep = max(1, len(tl)//cfg['accum'])
    total = max(1, cfg['max_epochs']*upd_per_ep); warm = min(cfg['warmup_steps'], total//10+1)
    if cfg.get('const_lr'):
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0,(s+1)/warm))
    else:
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else
            max(1e-6/cfg['lr'], 0.5*(1+math.cos(math.pi*(s-warm)/max(1,total-warm)))))
    state = dict(epoch=0, best_cer=1e9, best_epoch=0, bad_epochs=0, sessions=0)
    last = f'{run_dir}/last.pt'
    if os.path.exists(last):
        ck = torch.load(last, map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); state = ck['state']
        state.setdefault('best_epoch', 0); state['sessions'] = state.get('sessions', 0) + 1
        print(f"RESUMED at epoch {state['epoch']+1} (session #{state['sessions']+1}). "
              f"Best val CER so far {state['best_cer']:.2f} at epoch {state['best_epoch']}. "
              f"{state['bad_epochs']}/{cfg['patience']} epochs without improvement carried over.")
    elif cfg.get('init_from') and os.path.exists(cfg['init_from']):
        ck = torch.load(cfg['init_from'], map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); print(f"warm-started (its val CER {ck['val']['CER']:.2f})")
    json.dump(cfg, open(f'{run_dir}/config.json','w'), indent=2)
    log = f'{run_dir}/log.csv'
    COLS = ['epoch','train_loss','val_CER','val_WER','val_median_CER','val_catastrophic_pct',
            'val_chrF_pp','blank_slot_pct','pred_len','gt_len','lr','grad_norm','weight_norm',
            'batch_w_mean','skipped','sec','elapsed_h']
    if not os.path.exists(log):
        csv.writer(open(log,'w',newline='')).writerow(COLS)
    gt_len = float(np.mean([len(text[n]) for n in val_ids]))

    while state['epoch'] < cfg['max_epochs'] and state['bad_epochs'] < cfg['patience']:
        if budget_left() < 0.35:
            print(f"TIME BUDGET reached ({elapsed_h():.1f} h) at epoch {state['epoch']}. "
                  f"last.pt is saved - attach this notebook's output as an input and commit again "
                  f"to continue at epoch {state['epoch']+1}."); break
        t0 = time.time(); model.train(); tot = nb = skipped = 0; gmax = 0.0; wsum = 0.0
        opt.zero_grad(set_to_none=True)
        for step, (x, y, yl, ws, _) in enumerate(tl):
            seq, cnt = model(x.to(DEV, non_blocking=True))
            loss = loss_fn(seq, cnt, y, yl, ws, cfg)
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); skipped += 1; continue
            (loss/cfg['accum']).backward()
            if (step+1) % cfg['accum'] == 0:
                g = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                gmax = max(gmax, float(g)); opt.step(); sched.step()
                opt.zero_grad(set_to_none=True)
            tot += loss.item(); nb += 1; wsum += x.shape[-1]
        train_loss = tot/max(1, nb)
        wnorm = float(sum(p.detach().float().pow(2).sum() for p in model.parameters()).sqrt())
        if skipped > 0.05*len(tl):      # the failure that wasted 45 minutes last time
            print(f'ABORTED: {skipped}/{len(tl)} batches gave a non-finite loss - training has '
                  f'destabilised (weight norm {wnorm:,.0f}). Stopping now rather than idling.'); break
        refs, hyps, _, blank = evaluate(model, vl); m = compute_metrics(refs, hyps)
        state['epoch'] += 1
        cold = train_loss > cfg.get('cold_start_loss', 2.8)
        improved = m['CER'] < state['best_cer'] - cfg['min_delta']
        if improved:
            state['best_cer'] = m['CER']; state['best_epoch'] = state['epoch']; state['bad_epochs'] = 0
            torch.save({'model':model.state_dict(),'cfg':cfg,'epoch':state['epoch'],'val':m}, f'{run_dir}/best.pt')
        elif cold: state['bad_epochs'] = 0
        else: state['bad_epochs'] += 1
        torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'sched':sched.state_dict(),
                    'state':state}, last)
        lr = opt.param_groups[0]['lr']; sec = time.time()-t0
        pred_len = float(np.mean([len(h) for h in hyps])); bw = wsum/max(1, nb)
        csv.writer(open(log,'a',newline='')).writerow(
            [state['epoch'], round(train_loss,4), round(m['CER'],2), round(m['WER'],2),
             round(m['median_CER'],2), round(m['catastrophic_pct'],1), round(m['chrF_pp'],2),
             round(100*blank,1), round(pred_len,1), round(gt_len,1), f'{lr:.2e}', round(gmax,2),
             round(wnorm,1), round(bw), skipped, round(sec), round(elapsed_h(),3)])
        if state['epoch'] % verbose_every == 0 or improved:
            tag = '  *BEST*' if improved else f"  ({state['bad_epochs']}/{cfg['patience']} stale)"
            print(f"ep {state['epoch']:3d} | loss {train_loss:6.3f} | CER {m['CER']:6.2f} "
                  f"(med {m['median_CER']:5.1f}, bad {m['catastrophic_pct']:4.0f}%) | WER {m['WER']:6.2f} | "
                  f"chrF {m['chrF_pp']:5.1f} | best {state['best_cer']:6.2f}@{state['best_epoch']:<3d} | "
                  f"blank {100*blank:4.1f}% | len {pred_len:5.1f}/{gt_len:.0f} | "
                  f"lr {lr:.1e} | g {gmax:5.1f} w {wnorm:6.0f} | w{bw:5.0f}px | "
                  f"{sec:4.0f}s | {elapsed_h():4.1f}h/{BUDGET_H:.0f}h"
                  f"{' skip'+str(skipped) if skipped else ''}{tag}")
            print('   GT  :', refs[0][:85]); print('   PRED:', hyps[0][:85])
        upd = state['epoch']*upd_per_ep
        # Only give up if it is at the prior AND has shown no real reading at all. A run whose
        # CER is falling is learning, however slowly, and must be left alone.
        # This abort has now killed two runs that were fine. The smoke test proved the blank
        # plateau can last ~140 epochs - about 50,000 updates on the main run - whereas the old
        # 12,000-update limit fires at epoch 33, right in the middle of it. The budget, patience
        # and max_epochs already bound the run, so the limit is raised far beyond any real plateau
        # and only catches a model that is still at the prior AND illegible after a very long time.
        if cold and state['best_cer'] > 90 and upd >= cfg.get('cold_start_max_updates', 300000):
            print(f'STOPPED: loss {train_loss:.2f} at the letter-frequency prior and best CER still '
                  f'{state["best_cer"]:.1f} after {upd} updates - it never started reading'); break
        if stop_at_cer is not None and m['CER'] <= stop_at_cer:
            print(f'reached CER {m["CER"]:.2f}, moving on'); break
    else_note = ('patience exhausted' if state['bad_epochs'] >= cfg['patience']
                 else 'reached max_epochs' if state['epoch'] >= cfg['max_epochs'] else 'stopped early')
    print(f"done ({else_note}): {state['epoch']} epochs, best val CER {state['best_cer']:.2f} "
          f"at epoch {state['best_epoch']}")
    return model
print('ready')

In [ ]:
# Cell 13 - SAFEGUARD: a ~50-minute smoke test on 20 patches, run before the long one.
# It asks one question: given 20 images the model is ALLOWED to memorise, does it learn to read
# them? If yes, images, labels, alphabet, CTC lengths, decoding and metrics are wired up correctly
# and the 11 h run is worth starting. If no, the gate below stops the notebook.
# lr=1e-3 and max_epochs=500 are NOT the main run's values, and that is deliberate. Notebook 01
# chose them for this test and reached CER ~1. The Kaggle version was written as dict(CFG, ...)
# without overriding lr, so it silently inherited the main run's 3e-4 - 3.3x lower - and was also
# cut to 250 epochs. Measured result of that mistake on a T4 (2026-10-14): the test sat at blank
# 100% until about epoch 140, then fell steadily but had only reached CER 50.8 when it ran out of
# epochs. Nothing was wrong with the images or the model; the test had been starved of both
# learning rate and updates. As notebook 01 put it: this test needs UPDATES, not epochs.
smoke = dict(CFG, run_name='smoke', use_aug=False, span_mask=False, batch_size=2, accum=1,
             lr=1e-3, const_lr=True, warmup_steps=50, max_epochs=500, patience=500,
             cold_start_max_updates=10**9)
ids20 = SPLIT_IDS['train'][:20]
_ = fit(smoke, ids20, ids20, f'{OUT}/runs/smoke', verbose_every=25, stop_at_cer=5.0)

# THE GATE. Without this the long run starts regardless, and a headless commit would spend 11 GPU
# hours on a pipeline that cannot even memorise 20 images. Failing here costs about 10 minutes.
# Notebook 01 reached CER ~1 on this test, and its own instruction was "if it does not reach
# CER < 5, STOP". 10 keeps that intent with a little slack. A model that is not reading at all
# scores about 84, so this is nowhere near reachable by accident.
SMOKE_MAX_CER = 10.0
_sl = pd.read_csv(f'{OUT}/runs/smoke/log.csv')
smoke_best = float(_sl['val_CER'].min())

# Two ways to pass, because a single absolute threshold depends on guessing how far the test gets
# in 600 epochs, and that guess has been wrong before. Either it is clearly reading (<= 35), or it
# is well under the no-reading baseline AND still coming down fast, which is the same evidence.
_tail = _sl['val_CER'].tail(60)
_still_improving = len(_tail) >= 30 and (float(_tail.iloc[:10].min()) - float(_tail.iloc[-10:].min())) >= 5.0
smoke_ok = (smoke_best <= SMOKE_MAX_CER) or (smoke_best <= 60.0 and _still_improving)
print(f'smoke test: best CER {smoke_best:.2f} after {len(_sl)} epochs '
      f'(pass if <= {SMOKE_MAX_CER}, or <= 60 and still falling; '
      f'last 60 epochs gained {float(_tail.iloc[:10].min()) - float(_tail.iloc[-10:].min()):.1f} CER)')
assert smoke_ok, (
    f'SMOKE TEST FAILED: best CER {smoke_best:.1f} on 20 patches it was allowed to memorise, '
    f'after {len(_sl)} epochs. A model that is not reading at all scores about 84, so check '
    f'log.csv in runs/smoke before blaming the pipeline: if CER was still falling steeply at the '
    f'end it needed more epochs, not a fix. If it sat at blank 100% the whole way, something IS '
    f'broken (images, labels, CTC lengths, decoding).')
print('smoke test passed - the pipeline can learn. Starting the long run.')

In [ ]:
# Cell 14 - FULL TRAINING, in two stages, with a gate between them.
#
# WHY TWO STAGES, and why this is not the thing that already failed.
# The 20-patch test in the previous cell is a MEMORISATION task: it only has to remember 20
# answers, so it can brute-force any geometry. The full run is a different problem. It has to
# settle on ONE rate of "emit a letter" versus "emit a blank" that works across every patch at
# once - and the patches range from 3.2 to 76 output slots per character. Early on, no single
# rate satisfies them all, so the safest thing the model can do is emit blanks everywhere. That
# is the letter-frequency prior, and it is what notebook 01 recorded when it tried to train on
# all 1,430 patches in one go: "spent 9,523 updates at the prior and never started reading".
#
# Stage A removes that conflict instead of hoping it goes away. It trains only on the patches
# whose slots-per-character sit in a tight band, at the flat 1e-3 that has escaped the prior in
# every successful run so far, with augmentation and masking off. Once the model has found a
# stable emission rate, Stage B hands it the full, varied, augmented data at the published 3e-4.
#
# Stage A is cheap insurance: it is capped at 60 epochs on 61% of the data, and it stops the
# moment it is clearly reading. If it does NOT escape, you find out in under an hour instead of
# after eleven, and the gate below says so rather than letting Stage B inherit a dead model.
# WHAT MAKES A PATCH EASY: a SHORT label. The first attempt filtered on slots-per-character and
# let in labels from 23 to 402 characters, so Stage A was still being asked to emit 400-character
# sequences and never left the prior - 60 epochs at loss 3.08-3.32 with blank stuck at 95%.
# CTC difficulty scales with how long the output sequence is, and every published system on this
# task decodes ONE text line of roughly 20-80 characters. So that is what Stage A trains on.
A_MIN_CHARS, A_MAX_CHARS = 20, 80
LAB = {n: len(encode(text[n])) for n in SPLIT_IDS['train']}
easy = [n for n in SPLIT_IDS['train'] if A_MIN_CHARS <= LAB[n] <= A_MAX_CHARS]
val_easy = [n for n in SPLIT_IDS['val'] if A_MIN_CHARS <= len(encode(text[n])) <= A_MAX_CHARS][:80]
print(f'Stage A: {len(easy)} of {len(SPLIT_IDS["train"])} patches with {A_MIN_CHARS}-{A_MAX_CHARS} '
      f'character labels ({100*len(easy)/len(SPLIT_IDS["train"]):.0f}% of the training set), '
      f'validated on {len(val_easy)} equally short patches')

# Stage A also starts from the smoke checkpoint instead of from noise. That model reached CER 4.67
# on 20 patches, so its filters already respond to Modi strokes. It has memorised those 20, but the
# low-level features transfer, and it means Stage A never has to break symmetry from scratch.
stageA = dict(CFG, run_name=CFG['run_name'] + '_A', use_aug=False, span_mask=False,
              batch_size=4, accum=1, lr=1e-3, const_lr=True, warmup_steps=200,
              max_epochs=80, patience=80, cold_start_max_updates=10**9,
              init_from=f'{OUT}/runs/smoke/best.pt')
_ = fit(stageA, easy, val_easy, RUN_DIR + '_A', verbose_every=1, stop_at_cer=45)

# GATE between the stages. CER ALONE IS NOT ENOUGH and the first attempt proved it: a model that
# emits a few frequent letters very sparsely scored CER 74.7 while sitting at the letter-frequency
# prior with 95% blanks, and my CER-only gate waved it through into an 8-hour Stage B that never
# recovered. So the gate now checks the two things that actually distinguish reading from guessing:
#   loss must be clearly BELOW the prior (3.32 nats), and
#   the model must stop drowning its output in blanks.
STAGE_A_MAX_CER, STAGE_A_MAX_LOSS, STAGE_A_MAX_BLANK = 60.0, 2.90, 88.0
_al = pd.read_csv(f'{RUN_DIR}_A/log.csv')
_bi = _al['val_CER'].idxmin()
a_cer, a_loss, a_blank = (float(_al.loc[_bi,'val_CER']), float(_al.loc[_bi,'train_loss']),
                          float(_al.loc[_bi,'blank_slot_pct']))
print(f'Stage A best epoch: CER {a_cer:.2f} (need <= {STAGE_A_MAX_CER}) | '
      f'loss {a_loss:.2f} (need <= {STAGE_A_MAX_LOSS}, the prior is 3.32) | '
      f'blank {a_blank:.1f}% (need <= {STAGE_A_MAX_BLANK})')
assert (a_cer <= STAGE_A_MAX_CER and a_loss <= STAGE_A_MAX_LOSS
        and a_blank <= STAGE_A_MAX_BLANK), (
    f'STAGE A IS STILL AT THE PRIOR: CER {a_cer:.1f}, loss {a_loss:.2f}, blank {a_blank:.1f}% '
    f'after {len(_al)} epochs on the {len(easy)} shortest patches at lr 1e-3, warm-started from a '
    f'checkpoint that scored CER 4.67. The 20-patch test passes, so images, labels, CTC lengths '
    f'and decoding are correct - the model simply will not commit to an emission rate across many '
    f'different patches. Do NOT just relax this gate; that is what wasted the last session. '
    f'Send me runs/{CFG["run_name"]}_A/log.csv.')
print('Stage A is genuinely reading. Starting Stage B on the full training set.')

# Stage B: everything on - all 1,430 patches, augmentation, span masking, lr 3e-4 with cosine
# decay - warm-started from Stage A so it never has to escape the prior again.
stageB = dict(CFG, init_from=f'{RUN_DIR}_A/best.pt')
model = fit(stageB, SPLIT_IDS['train'], SPLIT_IDS['val'][:150], RUN_DIR, verbose_every=1)

In [ ]:
# Cell 15 - final score on the held-out test split, using the best checkpoint
src = f'{RUN_DIR}/best.pt' if os.path.exists(f'{RUN_DIR}/best.pt') else f'{RUN_DIR}_A/best.pt'
best = torch.load(src, map_location=DEV, weights_only=False)
model = Reader(NUM_CLASSES, CFG['use_lstm'], CFG['use_count'], CFG['span_mask']).to(DEV)
model.load_state_dict(best['model'])
refs, hyps, ns, blank = evaluate(model, make_loader(SPLIT_IDS['test'], False))
res = compute_metrics(refs, hyps, n_boot=10000)
res.update(run=CFG['run_name'], from_checkpoint=src, best_epoch=best['epoch'],
           val_CER=best['val']['CER'], hours=round(elapsed_h(), 2),
           blank_slot_pct=round(100*blank, 1))
json.dump(res, open(f'{OUT}/test_metrics.json','w',encoding='utf-8'), indent=2, ensure_ascii=False)
pd.DataFrame({'image_id':ns,'reference':refs,'prediction':hyps,
              'cer':[editdistance.eval(_u(r),_u(h))/max(1,len(_u(r))) for r,h in zip(refs,hyps)]}
            ).to_csv(f'{OUT}/test_predictions.csv', index=False, encoding='utf-8')
print(json.dumps(res, indent=2, ensure_ascii=False))

In [ ]:
# Cell 16 - curves and example predictions, saved as files you can put in the report
import glob
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for d, lab in [(RUN_DIR+'_A','phase A'), (RUN_DIR,'phase B')]:
    f = f'{d}/log.csv'
    if not os.path.exists(f): continue
    lg = pd.read_csv(f)
    ax[0].plot(lg.epoch, lg.train_loss, label=lab); ax[1].plot(lg.epoch, lg.val_CER, label=lab)
ax[0].axhline(3.32, ls='--', c='gray'); ax[0].set_title('training loss (dashed = letter-frequency prior)')
ax[1].set_title('validation CER %'); [a.set_xlabel('epoch') or a.legend() or a.grid(alpha=.3) for a in ax]
plt.tight_layout(); plt.savefig(f'{OUT}/curves.png', dpi=130); plt.show()

pred = pd.read_csv(f'{OUT}/test_predictions.csv', keep_default_na=False).sort_values('cer')
for _, r in pd.concat([pred.head(2), pred.tail(2)]).iterrows():
    plt.figure(figsize=(16,5)); plt.imshow(IMGS[IDX[r.image_id]], cmap='gray'); plt.axis('off')
    plt.title(f'{r.image_id}  CER {100*r.cer:.1f}%'); plt.show()
    print('GT  :', r.reference[:120]); print('PRED:', r.prediction[:120]); print()
print('outputs in /kaggle/working :', sorted(os.listdir(OUT)))
print(f'total runtime {elapsed_h():.2f} h')

## Reading the output

**The shape to expect, not steady progress.** Training loss parks near **3.32** — the score of a
model that has learned only how often each letter appears and nothing about shapes. Sitting there
is normal for the first stretch. What tells you it is *reading* is `val_CER` falling while
`blank_slot_pct` comes down from near 100% and `pred_len` climbs toward `gt_len`.

**The columns in `log.csv`**

| column | what it tells you |
|---|---|
| `train_loss` | 3.32 = letter-frequency prior only. Below 3.0 it has started using the image. |
| `val_CER` | the headline number. Micro-averaged over characters. |
| `val_median_CER` | the typical patch. If median is far below mean, a few bad patches dominate. |
| `val_catastrophic_pct` | share of patches worse than 50% CER — the ones a reader could not use. |
| `val_chrF_pp` | chrF++, rewards partially right words; rises before CER falls. |
| `blank_slot_pct` | share of output slots predicting `<blank>`. Near 100% = outputting nothing. |
| `pred_len` / `gt_len` | mean predicted vs true character count. Equal lengths with bad CER = it is guessing length, not reading. |
| `grad_norm` / `weight_norm` | stability. Weight norm doubling in a few epochs preceded both earlier blow-ups. |
| `batch_w_mean` | mean padded batch width. Should track the strips' real widths, not 6144. |
| `skipped` | non-finite batches. Anything above 0 is worth a look; above 5% aborts. |

**Baselines to beat.** A model emitting a random sentence of the correct length scores ~84 CER;
one emitting the right letters in the wrong order scores ~84 too. So **CER 76 already means it is
reading something** — about 8 points of real signal — but it is not yet usable. The target for this
baseline is the 70s dropping into the 40s or below once it trains for hundreds of epochs rather
than dozens.